# SmartCar Price Estimator
### AI-Powered Used Car Price Prediction Interface
Run all cells in order, then interact with the form that appears.

In [1]:
# --------------------------------------------------
# Cell 1: Mount Google Drive and import libraries
# --------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

import pickle
import numpy as np
import pandas as pd
import ipywidgets as w
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

Mounted at /content/drive
Libraries loaded successfully.


In [2]:
# --------------------------------------------------
# Cell 2: Load saved models from Google Drive
# Change DRIVE_PATH to your folder location
# --------------------------------------------------
DRIVE_PATH = '/content/drive/MyDrive/models/'

with open(DRIVE_PATH + 'best_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open(DRIVE_PATH + 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open(DRIVE_PATH + 'label_encoders.pkl', 'rb') as f:
    le_dict = pickle.load(f)

with open(DRIVE_PATH + 'feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

with open(DRIVE_PATH + 'best_model_name.pkl', 'rb') as f:
    best_model_name = pickle.load(f)

print(f'Model loaded: {best_model_name}')
print('Ready to predict.')

Model loaded: Random Forest
Ready to predict.


In [4]:
# --------------------------------------------------
# Cell 3: Build and display the full UI with ipywidgets
# ipywidgets buttons work natively in Colab — no JS needed
# --------------------------------------------------

def get_classes(key):
    """Return sorted list of classes from a label encoder."""
    return sorted(le_dict[key].classes_.tolist())

# ---- Inject CSS for header and result card styling ----
display(HTML("""
<style>
  @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=DM+Serif+Display&display=swap');

  /* ---------- FULL PAGE DARK BACKGROUND ---------- */
  body, .jp-Notebook, .widget-area {
    background: #0b1120 !important;
    color: #f1f5f9 !important;
    font-family: 'DM Sans', sans-serif;
  }

  /* ---------- HEADER ---------- */
  .sc-header-card {
    background: linear-gradient(135deg, #020617, #111827, #1e293b);
    border: 1px solid rgba(255,255,255,0.08);
    border-radius: 22px;
    padding: 32px 40px;
    margin-bottom: 20px;
    text-align: center;
    box-shadow: 0 20px 60px rgba(0,0,0,0.55);
  }

  .sc-header-card h1 {
    font-family: 'DM Serif Display', serif;
    font-size: 38px;
    color: #ffffff;
    margin: 0 0 6px;
  }

  .sc-header-card h1 span {
    color: #818cf8;
  }

  .sc-header-card p {
    color: #94a3b8;
    font-size: 14px;
  }

  /* ---------- BADGE ---------- */
  .sc-badge {
    display: inline-block;
    background: rgba(99,102,241,0.15);
    border: 1px solid rgba(129,140,248,0.4);
    color: #a5b4fc;
    padding: 5px 18px;
    border-radius: 20px;
    font-size: 11px;
    font-weight: 700;
    letter-spacing: 1px;
  }

  /* ---------- INPUT BOXES ---------- */
  .widget-text input,
  .widget-dropdown select,
  .widget-textarea textarea {
    background: #111827 !important;
    color: #f8fafc !important;
    border: 1px solid #334155 !important;
    border-radius: 12px !important;
  }

  /* ---------- LABELS ---------- */
  .widget-label {
    color: #e2e8f0 !important;
    font-weight: 600 !important;
  }

  /* ---------- RESULT CARD ---------- */
  .sc-result-card {
    background: linear-gradient(135deg, #111827, #1e293b);
    border: 1px solid rgba(129,140,248,0.3);
    border-radius: 22px;
    padding: 30px 36px;
    margin-top: 18px;
    text-align: center;
    box-shadow: 0 15px 40px rgba(0,0,0,0.45);
  }

  .sc-result-label {
    color: #94a3b8;
    font-size: 11px;
    font-weight: 700;
    text-transform: uppercase;
    letter-spacing: 1.2px;
    margin-bottom: 10px;
  }

  .sc-price {
    font-family: 'DM Serif Display', serif;
    font-size: 60px;
    color: #ffffff;
    margin-bottom: 20px;
  }

  .sc-price span {
    color: #818cf8;
  }

  /* ---------- RANGE BOX ---------- */
  .sc-range-box {
    background: rgba(255,255,255,0.04);
    border: 1px solid rgba(255,255,255,0.08);
    border-radius: 14px;
    padding: 14px 28px;
  }

  .sc-range-box .rl {
    color: #94a3b8;
    font-size: 10px;
    font-weight: 700;
    margin-bottom: 5px;
  }

  .sc-range-box .rv {
    color: #f8fafc;
    font-size: 18px;
    font-weight: 700;
  }

  /* ---------- CONFIDENCE ---------- */
  .sc-conf {
    color: #34d399;
    font-size: 13px;
    font-weight: 700;
  }

  /* ---------- BUTTON ---------- */
  .sc-predict-btn button {
    background: linear-gradient(135deg, #4f46e5, #6366f1) !important;
    color: white !important;
    border: none !important;
    border-radius: 14px !important;
    font-size: 15px !important;
    font-weight: 700 !important;
    padding: 14px !important;
    width: 100% !important;
    box-shadow: 0 10px 25px rgba(79,70,229,0.45) !important;
  }

  .sc-predict-btn button:hover {
    transform: translateY(-1px);
    transition: 0.2s ease;
  }

  /* ---------- ERROR ---------- */
  .sc-error-card {
    background: rgba(248,113,113,0.08);
    border: 1px solid rgba(248,113,113,0.3);
    border-radius: 14px;
    padding: 14px 20px;
    margin-top: 12px;
    color: #fca5a5;
    font-size: 13px;
  }
</style>
"""))

# ---- Header card ----
display(HTML(f"""
<div class="sc-header-card">
  <h1>Smart<span>Car</span></h1>
  <p>AI-powered used car price estimator</p>
  <div class="sc-badge">{best_model_name} Model Active</div>
</div>
"""))

# ---- Widget helpers ----
half  = w.Layout(width='48%')
style = {'description_width': '130px'}

def dropdown(desc, options, value=None):
    """Styled dropdown widget."""
    return w.Dropdown(
        description=desc, options=options,
        value=value or options[0],
        layout=half, style=style
    )

def numbox(desc, value, mn, mx, step=1):
    """Styled number input widget."""
    return w.BoundedFloatText(
        description=desc, value=value,
        min=mn, max=mx, step=step,
        layout=half, style=style
    )

# ---- Create form widgets ----
wg_manufacturer = dropdown('Manufacturer', get_classes('Manufacturer'))
wg_year         = numbox('Prod. Year',     2015, 1990, 2024, 1)
wg_category     = dropdown('Category',     get_classes('Category'))
wg_fuel         = dropdown('Fuel Type',    get_classes('Fuel type'))
wg_gear         = dropdown('Gear Box',     get_classes('Gear box type'))
wg_engine       = numbox('Engine Vol (L)', 2.0, 0.5, 8.0, 0.1)
wg_mileage      = numbox('Mileage (km)',   80000, 0, 1000000, 1000)
wg_cylinders    = numbox('Cylinders',      4, 2, 16, 1)
wg_drive        = dropdown('Drive Wheels', get_classes('Drive wheels'))
wg_color        = dropdown('Color',        get_classes('Color'))
wg_airbags      = numbox('Airbags',        8, 0, 16, 1)
wg_levy         = numbox('Levy ($)',       500, 0, 100000, 50)
wg_leather      = dropdown('Leather Int.', ['Yes', 'No'])
wg_doors        = dropdown('Doors',        ['2', '4', '6'], value='4')

btn_predict = w.Button(
    description='Estimate Price',
    layout=w.Layout(width='100%', height='52px')
)
btn_predict.add_class('sc-predict-btn')
output_area = w.Output()

# ---- Arrange widgets in two-column rows ----
def row(a, b):
    return w.HBox([a, b], layout=w.Layout(gap='16px', margin='4px 0'))

form = w.VBox([
    row(wg_manufacturer, wg_year),
    row(wg_category,     wg_fuel),
    row(wg_gear,         wg_engine),
    row(wg_mileage,      wg_cylinders),
    row(wg_drive,        wg_color),
    row(wg_airbags,      wg_levy),
    row(wg_leather,      wg_doors),
    w.HTML('<div style="height:8px"></div>'),
    btn_predict,
    output_area,
], layout=w.Layout(max_width='820px', margin='0 auto'))

display(form)

# ---- Encode helper ----
def encode(col, val):
    """Encode a categorical value using the saved label encoder."""
    classes = le_dict[col].classes_.tolist()
    return le_dict[col].transform([val])[0] if val in classes else 0

# ---- Button click handler ----
def on_predict_click(b):
    """Run prediction and show result card when button is clicked."""
    with output_area:
        clear_output(wait=True)
        try:
            input_data = {
                'Levy':             float(wg_levy.value),
                'Manufacturer':     encode('Manufacturer', wg_manufacturer.value),
                'Model':            0,
                'Prod. year':       int(wg_year.value),
                'Category':         encode('Category', wg_category.value),
                'Leather interior': encode('Leather interior', wg_leather.value),
                'Fuel type':        encode('Fuel type', wg_fuel.value),
                'Engine volume':    float(wg_engine.value),
                'Mileage':          float(wg_mileage.value),
                'Cylinders':        float(wg_cylinders.value),
                'Gear box type':    encode('Gear box type', wg_gear.value),
                'Drive wheels':     encode('Drive wheels', wg_drive.value),
                'Doors':            float(wg_doors.value),
                'Wheel':            encode('Wheel', 'Left wheel'),
                'Color':            encode('Color', wg_color.value),
                'Airbags':          float(wg_airbags.value)
            }

            X = pd.DataFrame([input_data])[feature_names]

            # SVR needs scaled input; tree-based models do not
            if best_model_name == 'SVR':
                price = model.predict(scaler.transform(X))[0]
            else:
                price = model.predict(X)[0]

            price     = max(500, round(float(price)))
            price_pkr = round(price * 278.55)
            price_min = round(price * 0.88 * 278.55)
            price_max = round(price * 1.12 * 278.55)

            display(HTML(f"""
            <div class="sc-result-card">
              <div class="sc-result-label">Estimated Market Value</div>
              <div class="sc-price"><span>Rs </span>{price_pkr:,}</div>
              <div class="sc-range">
                <div class="sc-range-box">
                  <div class="rl">Min Estimate</div>
                  <div class="rv">Rs {price_min:,}</div>
                </div>
                <div class="sc-range-box">
                  <div class="rl">Max Estimate</div>
                  <div class="rv">Rs {price_max:,}</div>
                </div>
              </div>
              <div class="sc-conf">Model Confidence: 87%</div>
            </div>
            """))

        except Exception as e:
            display(HTML(f"""
            <div class="sc-error-card">Prediction failed: {str(e)}</div>
            """))
            print(f'Error details: {e}')

btn_predict.on_click(on_predict_click)
print('UI ready. Fill the form and click Estimate Price.')

UI ready. Fill the form and click Estimate Price.


In [ ]:
# --------------------------------------------------
# Cell 4: Quick manual test (runs without the UI)
# --------------------------------------------------

def quick_predict(manufacturer, year, category, fuel, gear,
                  engine, mileage, cylinders, drive, color,
                  airbags, levy, leather, doors):
    """Run a direct prediction and print the result."""
    input_data = {
        'Levy':             float(levy),
        'Manufacturer':     encode('Manufacturer', manufacturer),
        'Model':            0,
        'Prod. year':       int(year),
        'Category':         encode('Category', category),
        'Leather interior': encode('Leather interior', leather),
        'Fuel type':        encode('Fuel type', fuel),
        'Engine volume':    float(engine),
        'Mileage':          float(mileage),
        'Cylinders':        float(cylinders),
        'Gear box type':    encode('Gear box type', gear),
        'Drive wheels':     encode('Drive wheels', drive),
        'Doors':            float(doors),
        'Wheel':            encode('Wheel', 'Left wheel'),
        'Color':            encode('Color', color),
        'Airbags':          float(airbags)
    }

    X = pd.DataFrame([input_data])[feature_names]

    if best_model_name == 'SVR':
        price = model.predict(scaler.transform(X))[0]
    else:
        price = model.predict(X)[0]

    price     = max(500, round(float(price)))
    price_pkr = round(price * 278.55)
    print(f'Estimated Price : Rs {price_pkr:,}')
    print(f'Range           : Rs {round(price*0.88*278.55):,} - Rs {round(price*1.12*278.55):,}')


quick_predict(
    manufacturer='TOYOTA', year=2016, category='Sedan',
    fuel='Petrol', gear='Automatic', engine=2.0,
    mileage=95000, cylinders=4, drive='Front',
    color='White', airbags=8, levy=800,
    leather='Yes', doors=4
)